In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

acct = QiskitRuntimeService.saved_accounts()["premium_new_usa"]

print("channel:", acct["channel"])
print("instance:", acct.get("instance"))
print("url:", acct.get("url"))

channel: ibm_quantum_platform
instance: crn:v1:bluemix:public:quantum-computing:us-east:a/cb804b30dfcb48b890393bfd6e41e9c2:e4da64d9-a9b9-407b-8a54-d981f9a61ffa::
url: https://cloud.ibm.com


In [9]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(name="basquecountry_updated")

print("Active instance:", service.active_instance())
print([b.name for b in service.backends(simulator=False, operational=True)])

Active instance: crn:v1:bluemix:public:quantum-computing:eu-de:a/cb804b30dfcb48b890393bfd6e41e9c2:f6803398-a738-4be7-95c0-140c91366c8f::
['ibm_basquecountry']


In [18]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

INSTANCE_NAME = "premium_new_usa"
JOB_ID = "fd9225f0-03b1-46f7-aaf1-f3c7884ff6e2"

catalog = QiskitFunctionsCatalog(name=INSTANCE_NAME)
qctrl_job = catalog.job(JOB_ID)

qctrl_job.status()

'QUEUED'

# Minimal Q-CTRL Sampler Demo

This notebook tries the simplest documented workaround first: do not force `backend_name`.
According to the IBM/Q-CTRL API docs, if `backend_name` is omitted, the function should choose the least busy backend available to your instance.


In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_catalog import QiskitFunctionsCatalog

INSTANCE_NAME = "premium_new_usa"
BACKEND_NAME = "ibm_pittsburgh"
SHOTS = 8

catalog = QiskitFunctionsCatalog(name=INSTANCE_NAME)
perf_mgmt = catalog.load("q-ctrl/performance-management")

qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

qctrl_job = perf_mgmt.run(
    primitive="sampler",
    pubs=[(qc, None, SHOTS)],
    backend_name=BACKEND_NAME,
    options={"job_tags": ["qctrl_smoke_premium_new_usa"]},
)

print("Q-CTRL job ID:", qctrl_job.job_id)
print("Status:", qctrl_job.status())

Q-CTRL job ID: 8699a7bf-3321-4497-8bb0-ab7646e01924
Status: QUEUED


In [20]:
status = str(qctrl_job.status()).upper()
print("Job ID:", qctrl_job.job_id)
print("Current status:", status)

if status == "DONE":
    print("The job has finished successfully.")
elif status in {"ERROR", "CANCELLED"}:
    print("The job has finished unsuccessfully.")
else:
    print("The job is still in progress.")


Job ID: 8699a7bf-3321-4497-8bb0-ab7646e01924
Current status: QUEUED
The job is still in progress.


In [ ]:
sampler_result = qctrl_job.result()

pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print(f"Counts for the measurement register: {counts}")


Cancela todos los jobs

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog
from qiskit_ibm_runtime import QiskitRuntimeService, Session

ACCOUNT_NAMES = ["premium_new_usa", "premium_new"]

ACTIVE_STATES = {
    "QUEUED",
    "PENDING",
    "RUNNING",
    "INITIALIZING",
    "MAPPING",
    "OPTIMIZING_HARDWARE",
    "WAITING_QPU",
    "EXECUTING_QPU",
    "POST_PROCESSING",
}

for account_name in ACCOUNT_NAMES:
    print(f"\n=== Cancelando jobs Q-CTRL en {account_name} ===")

    catalog = QiskitFunctionsCatalog(name=account_name)
    runtime_service = QiskitRuntimeService(name=account_name)
    perf_mgmt = catalog.load("q-ctrl/performance-management")

    jobs = perf_mgmt.jobs(limit=100)

    for job in jobs:
        status = str(job.status()).upper()

        if status not in ACTIVE_STATES:
            continue

        print(f"\nJob Q-CTRL activo: {job.job_id} [{status}]")

        # Si Q-CTRL ya creó sesiones Runtime, cancelarlas primero.
        try:
            session_ids = job.runtime_sessions()
        except Exception as exc:
            session_ids = []
            print("  No se pudieron consultar sesiones Runtime:", exc)

        for session_id in session_ids:
            try:
                session = Session.from_id(session_id, service=runtime_service)
                session.cancel()
                print("  Sesión Runtime cancelada:", session_id)
            except Exception as exc:
                print("  Error cancelando sesión", session_id, ":", exc)

        # Si Q-CTRL ya creó jobs Runtime, cancelarlos también directamente.
        try:
            runtime_job_ids = job.runtime_jobs()
        except Exception as exc:
            runtime_job_ids = []
            print("  No se pudieron consultar jobs Runtime:", exc)

        for runtime_job_id in runtime_job_ids:
            try:
                runtime_job = runtime_service.job(runtime_job_id)
                runtime_status = str(runtime_job.status()).upper()
                print(f"  Runtime job asociado: {runtime_job_id} [{runtime_status}]")

                if runtime_status not in {"DONE", "ERROR", "CANCELLED"}:
                    runtime_job.cancel()
                    print("  Runtime job cancelado:", runtime_job_id)
            except Exception as exc:
                print("  Error cancelando runtime job", runtime_job_id, ":", exc)

        # Cancelar el job contenedor de Q-CTRL / Qiskit Functions.
        try:
            message = job.cancel(service=runtime_service)
            print("  Cancelación Q-CTRL:", message)
        except Exception as exc:
            print("  Error cancelando job Q-CTRL:", type(exc).__name__, exc)


=== Cancelando jobs Q-CTRL en premium_new_usa ===

Job Q-CTRL activo: 0ab230d6-907b-4a02-bb33-25d9a11f8cbe [QUEUED]
  Cancelación Q-CTRL: Job has been stopped. No active runtime job ID associated with this serverless job ID.


KeyboardInterrupt: 

Verifica si quedan jobs activos

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

ACCOUNT_NAMES = ["premium_new_usa", "premium_new"]

ACTIVE_STATES = {
    "QUEUED",
    "PENDING",
    "RUNNING",
    "INITIALIZING",
    "MAPPING",
    "OPTIMIZING_HARDWARE",
    "WAITING_QPU",
    "EXECUTING_QPU",
    "POST_PROCESSING",
}

remaining_active_jobs = []

for account_name in ACCOUNT_NAMES:
    print(f"\n=== Verificación Q-CTRL en {account_name} ===")

    catalog = QiskitFunctionsCatalog(name=account_name)
    perf_mgmt = catalog.load("q-ctrl/performance-management")

    jobs = perf_mgmt.jobs(limit=100)

    for job in jobs:
        status = str(job.status()).upper()
        print(job.job_id, status)

        if status in ACTIVE_STATES:
            remaining_active_jobs.append((account_name, job.job_id, status))

if remaining_active_jobs:
    print("\nATENCIÓN: todavía quedan jobs Q-CTRL activos:")
    for account_name, job_id, status in remaining_active_jobs:
        print(account_name, job_id, status)
else:
    print("\nNo quedan jobs Q-CTRL activos en las cuentas comprobadas.")


=== Verificación Q-CTRL en premium_new_usa ===
0ab230d6-907b-4a02-bb33-25d9a11f8cbe CANCELED
a7e7b59d-4439-4334-82c8-ebcc572c7bf6 DONE
a8ef920f-1b1a-44b2-88da-24db4ca313ff DONE
bc0b5d96-146a-43a7-8bdc-acd9f017c408 DONE
caefbeb4-2ec0-4ad9-a856-480e158d437c DONE


KeyboardInterrupt: 

## Prueba mínima sólo QPU

In [12]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

ACCOUNT_NAME = "premium_new_usa"
BACKEND_NAME = "ibm_pittsburgh"
SHOTS = 8

# Carga directamente la instancia IBM Runtime, sin Q-CTRL.
service = QiskitRuntimeService(name=ACCOUNT_NAME)
backend = service.backend(BACKEND_NAME)

print("Backend:", backend.name)
print("Operativo:", backend.status().operational)
print("Jobs pendientes:", backend.status().pending_jobs)

# Circuito mínimo.
qc = QuantumCircuit(1)
qc.h(0)
qc.measure_all()

# SamplerV2 requiere circuito ISA para ejecución Runtime directa.
isa_qc = backend.target
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
)

isa_circuit = pm.run(qc)

sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=SHOTS)

print("Runtime job ID:", job.job_id())
print("Estado inicial:", job.status())

Backend: ibm_pittsburgh
Operativo: True
Jobs pendientes: 0
Runtime job ID: d8aqkbqvnmpc73br14i0
Estado inicial: QUEUED


In [13]:
result = job.result()

counts = result[0].data.meas.get_counts()
print("Counts:", counts)

Counts: {'0': 4, '1': 4}
